# Airfoil v1 — feature tutorial

This notebook tours every feature added in **airfoil v1**:

1. Instantiating the v1 problem and inspecting its design space / objectives / conditions
   (note the new sampled `temperature` condition).
2. Running a single CFD analysis with `simulate` (drag, lift).
3. Running an IPOPT shape optimization with `optimize` (drag-min at a target lift; lift as a
   one-sided inequality; ±0.10 FFD shape bounds).
4. Reading the scalar objective trajectory (`optisteps`) **and** the rich geometry/surface
   trajectory of every intermediate design (`optimization_trajectory`).
5. The advanced overrides: which surface fields are written (`surface_variables`) and overriding
   the internal ADflow solver schedule (`solver_options`).
6. Locating the per-run debugging artifacts (`opt.hst`, `IPOPT.out`, `final_abs_volume.npy`,
   tarred per-iteration files).
7. Generating a dataset at scale on Slurm.

> The v1 backend runs MACH-Aero (ADflow + pyOptSparse) inside the `mdolab/public` container, so the
> `simulate`/`optimize` cells need a working container runtime (Docker/Podman/Apptainer) and the
> container image. A full optimization takes several minutes. The inspection/plotting cells are cheap.

## Setup

Select the container runtime (on an HPC cluster this is typically Apptainer).

In [ ]:
import os

os.environ.setdefault("CONTAINER_RUNTIME", "apptainer")

# Point this at a local .npy of baseline airfoil coordinates (shape (N, P, 2) or (N, 2, P)).
# Leave as None to instead draw a baseline from the published Hugging Face dataset.
COORDS_FILE = None
DESIGN_INDEX = 0
MPICORES = 4

## 1. Instantiate and inspect the v1 problem

Note `temperature` is now part of the sampled conditions.

In [ ]:
from engibench.problems.airfoil.v1 import Airfoil

problem = Airfoil(seed=0)
print("version:", problem.version, "| dataset:", problem.dataset_id)
problem.objectives

In [ ]:
problem.design_space

In [ ]:
problem.conditions  # mach, reynolds, temperature, area_initial, area_ratio_min, cl_target

## 2. A baseline design

Load a baseline section, either from a local coordinates file or from the dataset.

In [ ]:
import numpy as np

if COORDS_FILE is not None:
    raw = np.load(COORDS_FILE, allow_pickle=True)
    coords = np.asarray(raw[DESIGN_INDEX], dtype=float)
    coords = coords if coords.shape[0] == 2 else coords.T  # noqa: PLR2004  -- want (2, P)
    design = {"coords": coords, "angle_of_attack": 2.5}
else:
    design, _ = problem.random_design()  # needs the published dataset

print("baseline coords:", np.asarray(design["coords"]).shape, "| alpha:", design["angle_of_attack"])

## 3. `simulate`: one CFD analysis

The flow condition now carries a `temperature`. Returns `[drag, lift]`.

In [ ]:
sim_config = {"mach": 0.5, "reynolds": 5.0e6, "temperature": 288.15}
drag, lift = problem.simulate(design, config=sim_config, mpicores=MPICORES)
print(f"drag={drag:.6f}  lift={lift:.6f}  L/D={lift / drag:.2f}")

## 4. `optimize`: IPOPT shape optimization

Drag-minimization at a target lift. In v1 the lift is a one-sided inequality
(`cl_target <= cl <= 1.2*cl_target`) and the FFD shape variables span ±0.10.
`area_initial` is the baseline area the (baseline-scaled) area constraint references.

In [ ]:
from engibench.problems.airfoil.utils import calc_area

opt_config = {
    "mach": 0.5,
    "reynolds": 5.0e6,
    "temperature": 288.15,
    "cl_target": 0.5,
    "area_ratio_min": 0.8,
    "area_initial": calc_area(design["coords"]),
}
opt_design, optisteps = problem.optimize(design, config=opt_config, mpicores=MPICORES)
print("optimized alpha:", round(float(opt_design["angle_of_attack"]), 4))
print("optimized coords:", np.asarray(opt_design["coords"]).shape)

### 4a. Objective trajectory (`optisteps`)

The scalar drag objective per optimizer iteration, from `opt.hst`.

In [ ]:
import matplotlib.pyplot as plt

obj = [float(np.ravel(s.obj_values)[0]) for s in optisteps]
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(len(obj)), obj, marker=".")
ax.set_xlabel("optimizer step")
ax.set_ylabel("drag coefficient")
ax.set_title("Objective trajectory")
plt.show()

### 4b. Geometry + surface trajectory (`optimization_trajectory`)

`optimization_trajectory()` returns the full geometry of **every evaluated design** (read from the
loose and tarred section files). With `include_surface=True`, each entry also carries a DataFrame of
every per-node surface field (cp, Mach, skin friction, separation sensors, …) — rich data for
trajectory/multimodal models.

In [ ]:
trajectory = problem.optimization_trajectory(include_surface=True)
print(f"{len(trajectory)} designs in the trajectory")
surf_last = trajectory[-1]["surface"]
print("surface fields:", list(surf_last.columns))

In [ ]:
# Overlay the baseline vs optimized geometry, and the optimized surface cp distribution.
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 3.2))

base = trajectory[0]["coords"]
opt = trajectory[-1]["coords"]
ax0.plot(base[0], base[1], label="baseline", lw=1)
ax0.plot(opt[0], opt[1], label="optimized", lw=1)
ax0.set_aspect("equal")
ax0.set_title("Airfoil: baseline vs optimized")
ax0.legend()

ax1.plot(surf_last["XoC"], surf_last["CoefPressure"], lw=1)
ax1.invert_yaxis()  # cp plots are conventionally drawn with -cp up
ax1.set_xlabel("x/c")
ax1.set_ylabel("cp")
ax1.set_title("Optimized surface pressure")
plt.show()

## 5. Advanced overrides: `surface_variables` and `solver_options`

The CFD solver schedule is applied internally so you don't have to tune it, but you can:
- restrict the written surface fields with `surface_variables`, and
- override any ADflow option (merged last) with `solver_options`.

In [ ]:
adv_config = {
    "mach": 0.5,
    "reynolds": 5.0e6,
    "temperature": 288.15,
    "surface_variables": ["cp", "mach", "yplus"],                # write only these surface fields
    "solver_options": {"nCycles": 5000, "L2Convergence": 1e-9},  # override the internal schedule
}
drag2, lift2 = problem.simulate(design, config=adv_config, mpicores=MPICORES)
print(f"(custom surface set + solver options) drag={drag2:.6f} lift={lift2:.6f}")

## 6. Debugging artifacts

Each run's `output/` directory is exposed via `study_output_dir`.

In [ ]:
output_dir = problem.study_output_dir
wanted = {"opt.hst", "IPOPT.out", "final_abs_volume.npy"}
artifacts = [f for f in sorted(os.listdir(output_dir)) if f in wanted or "intermediate" in f]
print("artifacts:", artifacts)
print("final absolute area:", np.load(os.path.join(output_dir, "final_abs_volume.npy")))

## 7. Dataset generation on Slurm

Generate a v1 dataset at scale with the optimize driver (samples mach / reynolds / temperature and
runs an IPOPT optimization per case):

```bash
python engibench/problems/airfoil/dataset_slurm_airfoil_optimize.py \
    -account <hpc_account> -n_designs 100 -n_flows 1 -group_size 4 \
    -min_ma 0.4 -max_ma 1.2 -min_re 1e6 -max_re 1e8 \
    -min_temp 220 -max_temp 310 -return_history \
    --coords_file /path/to/baseline_airfoils.npy
```